[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_06/3_stats.ipynb)

# Day 6: Statistics, A/B testing and product (easy)

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (25 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | A range for the tip rate | easy | 6 |
| Q2 | Did moving the gate change 7-day retention? | easy | 7 |
| Q3 | A range for the tip rate of card rides | easy | 7 |
| Q4 | How many rides to pin down the average tip? (review) | easy | 5 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

All real datasets (see DATASETS.md for sources). The setup cell loads them:

- `tips`: 244 restaurant bills (`total_bill`, `tip`, `sex`, `smoker`, `day`, `time`, `size`).
- `cookie`: Cookie Cats mobile game A/B test, 90,189 players. `version` is `gate_30` (control) or `gate_40` (treatment: the first gate moved from level 30 to 40); `sum_gamerounds` = rounds played in the first week; `retention_1`, `retention_7` = came back 1 / 7 days after install (bool).
- `taxis`: 6,433 New York taxi trips from March 2019 (`pickup`, `distance`, `fare`, `tip`, `total`, `payment`, `pickup_borough` ...).

Classic probability questions need no data: solve them on paper first, then check with a simulation (`rng = np.random.default_rng(0)`).

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import numpy as np
import pandas as pd
from scipy import stats as st
pd.set_option("display.width", 120)

tips = pd.read_csv(data_path("tips.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"))
cookie = pd.read_csv(data_path("cookie_cats.csv", "https://raw.githubusercontent.com/robguilarr/ab_testing_cookie_cats/main/datasets/cookie_cats.csv"))
taxis = pd.read_csv(data_path("taxis.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"), parse_dates=["pickup", "dropoff"])
for _name in ['tips', 'cookie', 'taxis']:
    print(_name, globals()[_name].shape)

---
### Q1. A range for the tip rate

*easy, about 6 min*

Using `tips`, let `tip_pct = 100 * tip / total_bill`.
1. Compute a 95% confidence interval for the mean tip percentage with the t distribution.
2. Which of these sentences is correct?
   - (a) 'There is a 95% probability that the true mean is inside this interval.'
   - (b) '95% of tables tip a percentage inside this interval.'
   - (c) 'If we repeated this sampling many times, 95% of intervals built this way would contain the true mean.'
3. Print the share of individual tables whose tip_pct falls inside the interval.

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# tips is loaded in the setup

<details><summary><b>Hint 1</b></summary>

Signal: a range for a mean from one sample with unknown sd. Topic: the t confidence interval `mean +- t(0.975, n - 1) * sd / sqrt(n)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. `x = 100 * tips.tip / tips.total_bill`; `se = x.std() / np.sqrt(len(x))`. 2. `t = st.t.ppf(0.975, len(x) - 1)`. 3. Interval = `x.mean() +- t * se` (or `st.t.interval(0.95, df, loc, scale)`). 4. Count tables between the two limits.

</details>

<details><summary><b>Solution</b></summary>

```python
x = 100 * tips["tip"] / tips["total_bill"]
n, m, se = len(x), x.mean(), x.std() / np.sqrt(len(x))
t = st.t.ppf(0.975, n - 1)
lo, hi = m - t * se, m + t * se
print(f"n {n}  mean {m:.2f}  SE {se:.3f}  t {t:.3f}")
print(f"95% CI for the mean tip %: [{lo:.2f}, {hi:.2f}]")
print(f"share of tables inside the CI: {((x >= lo) & (x <= hi)).mean():.3f}")

# Output:
# n 244  mean 16.08  SE 0.391  t 1.970
# 95% CI for the mean tip %: [15.31, 16.85]
# share of tables inside the CI: 0.127
```

**Why:** The CI describes uncertainty about the *mean*, so it is narrow (width about 1.5 points) while single tables vary a lot: only about 1 in 8 tables (12.7%) tips inside it. That shows (b) is wrong: a range for individual values is a *prediction* interval, roughly `mean +- 2 sd`. Sentence (c) is the correct frequentist meaning. Sentence (a) is the common Bayesian-sounding reading; strictly, the true mean is fixed and the interval is random, but in practice examiners accept 'we are 95% confident' as long as you can explain (c). The t interval assumes roughly normal sample means: with n = 244 and moderate skew (one 71% tip), that holds well enough; a bootstrap would be a good robustness check.

**Say it to a PM:** On average, people tip between about 15.3% and 16.9% of the bill, and we are 95% confident the true average is in that range. Individual tables vary much more, so do not use this range to predict one table's tip.

**Common mistakes:** Choosing (a) or (b). Using 1.96 instead of the t quantile for small samples (here they are close: 1.970). Using the sd instead of the SE, which gives a prediction range, not a CI.

**Learn more:** [stats-confidence-intervals](https://mahsa-agz.github.io/ds-handbook/#stats-confidence-intervals)

</details>

---
### Q2. Did moving the gate change 7-day retention?

*easy, about 7 min*

In `cookie`, the treatment `gate_40` moved the first gate from level 30 to level 40.
1. Print the 7-day retention rate and the number of players in each version.
2. Compute a 95% confidence interval for the difference `gate_40 - gate_30` with the normal approximation (unpooled SE).
3. Also print the relative change in percent. Does the interval include 0? What would you tell the game team?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# cookie is loaded in the setup

<details><summary><b>Hint 1</b></summary>

Signal: two independent groups, a 0/1 metric, 'how big is the difference'. Topic: CI for a difference of two proportions, `SE = sqrt(p1 (1 - p1) / n1 + p2 (1 - p2) / n2)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. `g = cookie.groupby('version').retention_7.agg(['mean', 'count'])`. 2. `diff = p40 - p30`. 3. `se = sqrt(p30 (1 - p30) / n30 + p40 (1 - p40) / n40)`. 4. `diff +- 1.96 * se`. 5. Relative change = `diff / p30`.

</details>

<details><summary><b>Solution</b></summary>

```python
g = cookie.groupby("version")["retention_7"].agg(["mean", "count"])
(p30, n30), (p40, n40) = g.loc["gate_30"], g.loc["gate_40"]
diff = p40 - p30
se = np.sqrt(p30 * (1 - p30) / n30 + p40 * (1 - p40) / n40)
lo, hi = diff - 1.96 * se, diff + 1.96 * se
print(f"gate_30: {p30:.4f} (n={n30:.0f})   gate_40: {p40:.4f} (n={n40:.0f})")
print(f"difference {100 * diff:.2f} pp   95% CI [{100 * lo:.2f}, {100 * hi:.2f}] pp")
print(f"relative change {100 * diff / p30:.1f}%")

# Output:
# gate_30: 0.1902 (n=44700)   gate_40: 0.1820 (n=45489)
# difference -0.82 pp   95% CI [-1.33, -0.31] pp
# relative change -4.3%
```

**Why:** The two groups are independent random samples, so the variance of the difference is the sum of the two variances. The whole interval is below 0: the data are not compatible with 'no effect' at the 5% level, and they also tell us the size, roughly 0.3 to 1.3 points lower retention. A CI is more useful than a p-value alone because it shows the size and the uncertainty in the units the team cares about. The normal approximation is excellent here (about 45,000 players per arm and thousands of retained players in each).

**Say it to a PM:** Moving the gate to level 40 lowered 7-day retention from 19.0% to 18.2%, a drop of about 0.8 points (4% relative), and we are 95% confident the true drop is between about 0.3 and 1.3 points. Keep the gate at level 30.

**Common mistakes:** Adding the standard errors instead of the variances. Using the pooled SE for a CI (pooled is for the test under H0). Reporting only 'significant' without the size. Mixing absolute points and relative percent.

**Learn more:** [stats-confidence-intervals](https://mahsa-agz.github.io/ds-handbook/#stats-confidence-intervals), [stats-ab-analysis](https://mahsa-agz.github.io/ds-handbook/#stats-ab-analysis)

</details>

---
### Q3. A range for the tip rate of card rides

*easy, about 7 min*

For taxi rides paid by credit card (`taxis.payment == 'credit card'`), the finance team defines the tip rate as `sum(tip) / sum(fare)` (a ratio of two sums).
1. Compute it.
2. Build a 95% percentile bootstrap CI with 2,000 resamples of rides (`rng = np.random.default_rng(0)`).
3. Why is the bootstrap convenient here?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# taxis is loaded in the setup
card = taxis[taxis.payment == 'credit card']

<details><summary><b>Hint 1</b></summary>

Signal: a statistic with no easy SE formula (a ratio of sums). Topic: the bootstrap: resample the units with replacement, recompute the statistic, take the 2.5% and 97.5% percentiles.

</details>

<details><summary><b>Hint 2</b></summary>

1. `tip, fare = card.tip.to_numpy(), card.fare.to_numpy()`. 2. `idx = rng.integers(0, n, size=(2000, n))`. 3. `boot = tip[idx].sum(axis=1) / fare[idx].sum(axis=1)`. 4. `np.percentile(boot, [2.5, 97.5])`.

</details>

<details><summary><b>Solution</b></summary>

```python
card = taxis[taxis["payment"] == "credit card"]
tip, fare = card["tip"].to_numpy(), card["fare"].to_numpy()
n = len(card)
rate = tip.sum() / fare.sum()
rng = np.random.default_rng(0)
idx = rng.integers(0, n, size=(2000, n))          # each row = one bootstrap sample of rides
boot = tip[idx].sum(axis=1) / fare[idx].sum(axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"rides {n}   tip rate {100 * rate:.2f}%")
print(f"bootstrap SE {100 * boot.std():.2f} pp   95% CI [{100 * lo:.2f}%, {100 * hi:.2f}%]")

# Output:
# rides 4577   tip rate 20.31%
# bootstrap SE 0.23 pp   95% CI [19.86%, 20.78%]
```

**Why:** A ratio of sums is not a mean of independent values, so `sd / sqrt(n)` does not apply directly; the exact alternative is the delta method. The bootstrap avoids the algebra: it treats the sample as the population, resamples whole rides (keeping each ride's tip and fare together), and reads the spread of the recomputed statistic. Resample the independent unit: here the ride; in an A/B test, the user. The percentile method works well for smooth statistics with large n; it is less reliable for extremes (max, p99) and tiny samples.

**Say it to a PM:** Card-paying riders tip about 20.3% of the fare, and the plausible range is roughly 19.9% to 20.8%. That is a precise number we can use in the driver earnings model.

**Common mistakes:** Resampling tips and fares separately (it breaks the pairs). Averaging per-ride tip percentages and calling it the same metric (that weights short rides more). Using too few resamples (under 1,000) for a 95% interval.

**Learn more:** [stats-confidence-intervals](https://mahsa-agz.github.io/ds-handbook/#stats-confidence-intervals), [stats-ratio-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-ratio-metrics)

</details>

---
### Q4. How many rides to pin down the average tip? (review)

*easy, about 5 min*

Review of the standard error. Using all rides in `taxis`:
1. Print the mean and sd of `tip` and the SE of the mean.
2. How many rides would we need for a 95% margin of error of +- 5 cents (0.05)?
3. If we doubled the number of rides in this sample, by what factor would the margin shrink?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# taxis is loaded in the setup

<details><summary><b>Hint 1</b></summary>

Signal: 'how many observations for a given precision'. Topic: the square-root law, `margin = 1.96 * sd / sqrt(n)`, so `n = (1.96 * sd / margin)^2`.

</details>

<details><summary><b>Hint 2</b></summary>

1. `sd = taxis.tip.std()`, `se = sd / np.sqrt(len(taxis))`. 2. `n = (1.96 * sd / 0.05) ** 2`, round up. 3. Doubling n divides the margin by `sqrt(2)`.

</details>

<details><summary><b>Solution</b></summary>

```python
x = taxis["tip"]
sd, n = x.std(), len(x)
print(f"mean {x.mean():.3f}   sd {sd:.3f}   SE {sd / np.sqrt(n):.4f}   margin now +- {1.96 * sd / np.sqrt(n):.3f}")
print(f"rides for +- 0.05: {int(np.ceil((1.96 * sd / 0.05) ** 2)):,}")
print(f"doubling n shrinks the margin by a factor {np.sqrt(2):.3f}")

# Output:
# mean 1.979   sd 2.449   SE 0.0305   margin now +- 0.060
# rides for +- 0.05: 9,213
# doubling n shrinks the margin by a factor 1.414
```

**Why:** The margin is `1.96 * sd / sqrt(n)`. To make it a target size, solve for n: it grows with the square of `sd / margin`. Halving the margin costs 4 times the data, and doubling the data only shrinks the margin by 29% (factor 1.414). This is the same arithmetic behind A/B sample sizes (day 10).

**Say it to a PM:** Today we know the average tip to within about 6 cents. To get it within 5 cents we would need roughly 9,200 rides instead of 6,433.

**Common mistakes:** Forgetting to square. Using the SE where the sd belongs in the sample-size formula. Rounding n down.

**Learn more:** [stats-clt-se](https://mahsa-agz.github.io/ds-handbook/#stats-clt-se), [stats-power-mde](https://mahsa-agz.github.io/ds-handbook/#stats-power-mde)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_06/4_ai.ipynb)